In [1]:
from biomni.tool import hpc_search_tools
res = hpc_search_tools("HISAT2 alignment")
for r in res:
    print(r.get("id"), "|", r.get("name"))
    print(r.get("usage", "")[:1200])
    print("---")

hisat2 | HISAT2
HISAT2 - Graph-based Alignment of NGS Reads

CONTAINER ENVIRONMENT:
- Commands: hisat2, hisat2-build, hisat2-inspect (in PATH)
- Input files: /input (reference genome, FASTQ reads)
- Output files: /output (SAM alignments)

MAIN COMMANDS:

1. BUILD INDEX (basic):
hisat2-build -p 4   /input/reference.fa   /output/genome_index

2. BUILD INDEX (with splice sites):
hisat2-build -p 4   --exon /input/exons.txt   --ss /input/splice_sites.txt   /input/reference.fa   /output/genome_tran

3. PAIRED-END RNA-SEQ ALIGNMENT:
hisat2 -p 4   -x /input/genome_index   -1 /input/reads_1.fq   -2 /input/reads_2.fq   -S /output/alignment.sam

4. SINGLE-END ALIGNMENT:
hisat2 -p 4   -x /input/genome_index   -U /input/reads.fq   -S /output/alignment.sam

5. DNA ALIGNMENT (no splicing):
hisat2 -p 4   --no-spliced-alignment   -x /input/genome_index   -1 /input/reads_1.fq   -2 /input/reads_2.fq   -S /output/alignment.sam

KEY PARAMETERS:
-p/--threads INT          Number of threads (default: 1)
-x IN

In [2]:
from biomni.tool import hpc_run_tool
job = hpc_run_tool(
    "hisat2",
    "hisat2-build -p 8 /input/TAIR10.fa /output/TAIR10_index",
    input_files={"TAIR10.fa": "/mnt/results/chromdecode_at/tmp/TAIR10.fa"})
print(job)

{'job_id': 'f5bab3b2-7cc4-4737-9e04-c961d9cf70ed', 'status': 'downloading_inputs', 'created_at': '2026-10-01T14:32:18.373785+00:00', 'completed_at': None, 'tool_id': 'hisat2', 'error_logs': None, 'error_message': None}


In [3]:
from biomni.tool import hpc_get_job_results
res = hpc_get_job_results("f5bab3b2-7cc4-4737-9e04-c961d9cf70ed")
print(res.get("files"))
print(res.get("output_dir"))

{'outputs/TAIR10_index.1.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.1.ht2', 'outputs/TAIR10_index.2.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.2.ht2', 'outputs/TAIR10_index.3.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.3.ht2', 'outputs/TAIR10_index.4.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.4.ht2', 'outputs/TAIR10_index.5.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.5.ht2', 'outputs/TAIR10_index.6.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.6.ht2', 'outputs/TAIR10_index.7.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.7.ht2', 'outputs/TAIR10_index.8.ht2': '/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/outputs/TAIR10_index.8.ht2'}
/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed


In [4]:
from biomni.tool import hpc_run_tool
import time
samples = {
    "SRR6919520": "input_20min", "SRR6919521": "input_1h",
    "SRR6919522": "IP_SOG1_20min", "SRR6919523": "IP_SOG1_1h",
    "SRR6919524": "IP_wt_20min", "SRR6919526": "IP_wt_1h"}
idx = {f"TAIR10_index.{i}.ht2":
       f"/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/"
       f"outputs/TAIR10_index.{i}.ht2" for i in range(1, 9)}
jobs = {}
for srr, name in samples.items():
    files = dict(idx)
    files[f"{srr}.fastq.gz"] = (f"/mnt/results/chromdecode_at/tmp/"
                                f"GLDS-496_chip-seq_{srr}.fastq.gz")
    job = hpc_run_tool(
        "hisat2",
        f"hisat2 -p 8 --no-spliced-alignment -x /input/TAIR10_index "
        f"-U /input/{srr}.fastq.gz -S /output/{name}.sam",
        input_files=files)
    jobs[name] = job["job_id"]
    print(name, job["job_id"], job["status"])
    time.sleep(2)
print(jobs)

input_20min b4b2d726-601a-4642-bf8c-801d8a1bbf7f downloading_inputs


input_1h 964f3419-5877-49e9-a65c-6cf593a097d4 downloading_inputs


IP_SOG1_20min 8d821cc0-75a2-4d5b-b84e-ebe626bfca22 downloading_inputs


HPCAPIError: HPC API job creation failed (429 Too Many Requests): You already have 3 active CPU jobs running. Each user is limited to 3 concurrent CPU jobs. Please wait for a job to complete before submitting another.

In [5]:
from biomni.tool import hpc_run_tool
import time
remaining = {
    "SRR6919523": "IP_SOG1_1h", "SRR6919524": "IP_wt_20min",
    "SRR6919526": "IP_wt_1h"}
idx = {f"TAIR10_index.{i}.ht2":
       f"/mnt/results/hpc_f5bab3b2-7cc4-4737-9e04-c961d9cf70ed/"
       f"outputs/TAIR10_index.{i}.ht2" for i in range(1, 9)}
jobs = {}
for srr, name in remaining.items():
    files = dict(idx)
    files[f"{srr}.fastq.gz"] = (f"/mnt/results/chromdecode_at/tmp/"
                                f"GLDS-496_chip-seq_{srr}.fastq.gz")
    job = hpc_run_tool(
        "hisat2",
        f"hisat2 -p 8 --no-spliced-alignment -x /input/TAIR10_index "
        f"-U /input/{srr}.fastq.gz -S /output/{name}.sam",
        input_files=files)
    jobs[name] = job["job_id"]
    print(name, job["job_id"], job["status"])
    time.sleep(2)
print(jobs)

IP_SOG1_1h 9eb48256-ca9e-4320-8c90-f1eef1ddb4bf downloading_inputs


IP_wt_20min bea163c1-f03d-40c5-9b23-49e10183435b downloading_inputs


IP_wt_1h 73ccc578-9868-4588-8d48-33a3e0055ee7 downloading_inputs


{'IP_SOG1_1h': '9eb48256-ca9e-4320-8c90-f1eef1ddb4bf', 'IP_wt_20min': 'bea163c1-f03d-40c5-9b23-49e10183435b', 'IP_wt_1h': '73ccc578-9868-4588-8d48-33a3e0055ee7'}


In [6]:
from biomni.tool import hpc_get_job_results
for jid, name in [("b4b2d726-601a-4642-bf8c-801d8a1bbf7f", "input_20min"),
                 ("964f3419-5877-49e9-a65c-6cf593a097d4", "input_1h"),
                 ("8d821cc0-75a2-4d5b-b84e-ebe626bfca22", "IP_SOG1_20min")]:
    res = hpc_get_job_results(jid)
    print(name, "->", res.get("output_dir"),
          list(res.get("files", {}).keys())[:3])

input_20min -> /mnt/results/hpc_b4b2d726-601a-4642-bf8c-801d8a1bbf7f ['outputs/input_20min.sam']


input_1h -> /mnt/results/hpc_964f3419-5877-49e9-a65c-6cf593a097d4 ['outputs/input_1h.sam']


IP_SOG1_20min -> /mnt/results/hpc_8d821cc0-75a2-4d5b-b84e-ebe626bfca22 ['outputs/IP_SOG1_20min.sam']


In [7]:
from biomni.tool import hpc_get_job_results
for jid, name in [("9eb48256-ca9e-4320-8c90-f1eef1ddb4bf", "IP_SOG1_1h"),
                 ("73ccc578-9868-4588-8d48-33a3e0055ee7", "IP_wt_1h")]:
    res = hpc_get_job_results(jid)
    print(name, "->", res.get("output_dir"))

IP_SOG1_1h -> /mnt/results/hpc_9eb48256-ca9e-4320-8c90-f1eef1ddb4bf


IP_wt_1h -> /mnt/results/hpc_73ccc578-9868-4588-8d48-33a3e0055ee7


In [8]:
from biomni.tool import hpc_get_job_results
res = hpc_get_job_results("bea163c1-f03d-40c5-9b23-49e10183435b")
print("IP_wt_20min ->", res.get("output_dir"))

IP_wt_20min -> /mnt/results/hpc_bea163c1-f03d-40c5-9b23-49e10183435b
